<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/check_batch_dynamic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Batch Dynamics Check — Single City

This notebook loads a single city's delivery CSV, sorts by `delivery_user_id` and `receipt_time`, computes batch features (batch size, dispatch rank, batch id) and checks for `dynamic pickup` events: where a courier receives a new batch while some orders from the previous batch remain undelivered (sign_time > next_batch_receipt_time).

Edit `CITY` and `CITY_CSV_PATHS` below to point to your local files if needed.

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Imports and config
import os
import sys
from datetime import datetime
import numpy as np
import pandas as pd

# Select city to analyse (case-sensitive, e.g. 'Shanghai')
CITY = 'Shanghai'

# Candidate paths (edit if your files live elsewhere)
CITY_CSV_PATHS = [
    '/content/drive/MyDrive/ml/PROCESSED/matched/city_divided/shanghai_data.csv',
]

def find_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

CITY_CSV = find_existing_path(CITY_CSV_PATHS)
if CITY_CSV is None:
    print('No candidate CSV found. Please update CITY_CSV_PATHS to point to your file.')
    sys.exit(1)

print('Using CSV:', CITY_CSV)


Using CSV: /content/drive/MyDrive/ml/PROCESSED/matched/city_divided/shanghai_data.csv


In [11]:
# Load the CSV with pandas (robust for various environments)
df = pd.read_csv(CITY_CSV)

# Ensure datetime columns parsed
for col in ['receipt_time', 'sign_time']:
    if col in df.columns and not np.issubdtype(df[col].dtype, np.datetime64):
        df[col] = pd.to_datetime(df[col], format='%Y-%m-%d %H:%M:%S', errors='coerce')

# Derive or ensure 'ds' (MMDD) exists. If receipt_time contains full datetime, use it;
# otherwise rely on an existing 'ds' column in the CSV.
if 'ds' not in df.columns:
    if 'receipt_time' in df.columns:
        df['ds'] = df['receipt_time'].dt.strftime('%m%d')
    else:
        df['ds'] = pd.NA
# Normalize ds to string (zero-padded MMDD)
df['ds'] = df['ds'].astype(str)

# Basic cleaning: drop rows missing key timestamps or courier id
df = df.dropna(subset=['receipt_time', 'delivery_user_id'])
df['delivery_user_id'] = df['delivery_user_id'].astype(str)

# Compute eta_mins if present/needed
if 'sign_time' in df.columns:
    df['eta_mins'] = (df['sign_time'] - df['receipt_time']).dt.total_seconds() / 60

# Sort as requested: by delivery_user_id then ds then receipt_time then order_id (if exists)
sort_cols = ['delivery_user_id', 'ds', 'receipt_time']
if 'order_id' in df.columns:
    sort_cols.append('order_id')
df = df.sort_values(sort_cols).reset_index(drop=True)

print('Loaded rows:', len(df))
df.head(3)


Loaded rows: 34735


,order_id,from_dipan_id,delivery_user_id,poi_lng,poi_lat,aoi_id,typecode,receipt_time,receipt_lng,receipt_lat,sign_time,sign_lng,sign_lat,ds,city,eta_mins
0,04fc2f9b94c6de1069d525e259ca7d82,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940,1.056095e+07,-7.454289e+06,8cd1cc14a0e53f305b80dbae07983ada,203ac3454d75e02ebb0a3c6f51d735e4,2021-03-18 07:24:00,1.056254e+07,-7.452504e+06,2021-03-18 08:29:00,NaN,NaN,318,Shanghai,65.0
1,0a11f8e3fee958aa3df8e7ceab8a5146,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940,1.056086e+07,-7.454063e+06,d4a631a1f2165ab095adb674382b3eea,203ac3454d75e02ebb0a3c6f51d735e4,2021-03-18 07:24:00,1.056254e+07,-7.452508e+06,2021-03-18 08:47:00,NaN,NaN,318,Shanghai,83.0
2,c1039a5e963e50ac59b037f9f0c7a3ac,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940,1.056094e+07,-7.454282e+06,8cd1cc14a0e53f305b80dbae07983ada,203ac3454d75e02ebb0a3c6f51d735e4,2021-03-18 07:24:00,1.056252e+07,-7.452511e+06,2021-03-18 08:09:00,NaN,NaN,318,Shanghai,45.0


In [21]:
# Refined Batching: Group by courier, day, depot, and a 5-minute time window
WINDOW = '5min'

# Ensure receipt_time is the index for resampler-like grouping if needed,
# but here we use a simpler 'floor' approach for grouping.
df['receipt_time_window'] = df['receipt_time'].dt.floor(WINDOW)

group_cols = ['delivery_user_id', 'ds', 'from_dipan_id', 'receipt_time_window']

if 'order_id' in df.columns:
    df['batch_size'] = df.groupby(group_cols)['order_id'].transform('count')
else:
    df['batch_size'] = df.groupby(group_cols).transform('size')

# Updated batch_id to include the time window
df['batch_id'] = df['delivery_user_id'] + '__' + df['ds'].astype(str) + '__' + \
                 df['receipt_time_window'].astype('int64').floordiv(10**9).astype(str) + '__' + \
                 df['from_dipan_id'].astype(str)

if 'order_id' in df.columns:
    df['batch_rank_dispatch'] = df.groupby(group_cols)['order_id'].cumcount()
else:
    df['batch_rank_dispatch'] = df.groupby(group_cols).cumcount()

if 'sign_time' in df.columns:
    df['batch_rank_actual'] = df.groupby(group_cols)['sign_time'].rank(method='first').astype('Int64') - 1
else:
    df['batch_rank_actual'] = pd.NA

print(f'Batch features computed using a {WINDOW} window. Example:')
display(df[['delivery_user_id','ds','receipt_time','from_dipan_id','batch_id','batch_size']].head(8))

Batch features computed using a 5min window. Example:


,delivery_user_id,ds,receipt_time,from_dipan_id,batch_id,batch_size
0,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:24:00,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940__318__1616052...,4
1,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:24:00,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940__318__1616052...,4
2,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:24:00,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940__318__1616052...,4
3,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:24:00,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940__318__1616052...,4
4,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:25:00,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940__318__1616052...,8
5,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:25:00,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940__318__1616052...,8
6,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:25:00,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940__318__1616052...,8
7,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:25:00,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940__318__1616052...,8


In [22]:
# Check for dynamic pickup per day with warehouse-switch constraint
# Condition: t_new_receipt < max(t_i_sign) AND from_dipan_id_new != from_dipan_id_old

if 'sign_time' not in df.columns:
    print('sign_time column missing — cannot compute dynamic pickup.')
else:
    ds_values = sorted(df['ds'].dropna().unique())
    if len(ds_values) == 0: ds_values = [None]

    for ds_val in ds_values:
        print(f'\n=== Day: {ds_val} ===')
        df_day = df[df['ds'] == ds_val].copy() if ds_val else df.copy()
        if df_day.empty: continue

        # Build batch summary including the depot ID
        batches = df_day.groupby(['delivery_user_id','batch_id'], as_index=False).agg({
            'receipt_time': 'first',
            'from_dipan_id': 'first',
            'batch_size': 'first'
        })
        batches = batches.sort_values(['delivery_user_id','receipt_time']).reset_index(drop=True)

        # Shift to get next batch info per courier
        batches['next_receipt'] = batches.groupby('delivery_user_id')['receipt_time'].shift(-1)
        batches['next_dipan'] = batches.groupby('delivery_user_id')['from_dipan_id'].shift(-1)

        def check_dynamic_pickup(row):
            if pd.isna(row['next_receipt']) or pd.isna(row['next_dipan']):
                return 0
            # Condition 1: Warehouse ID must be different
            if row['from_dipan_id'] == row['next_dipan']:
                return 0
            # Condition 2: Next batch arrived before current batch was fully signed
            mask = (df_day['batch_id'] == row['batch_id']) & (df_day['sign_time'] > row['next_receipt'])
            return 1 if mask.any() else 0

        batches['dynamic_pickup_flag'] = batches.apply(check_dynamic_pickup, axis=1)

        total_batches = len(batches)
        dyn_count = int(batches['dynamic_pickup_flag'].sum())
        print(f'Total batches: {total_batches:,}')
        print(f'Dynamic Pickups (Cross-Depot Overlap): {dyn_count:,} ({dyn_count/total_batches*100:.2f} %)')

        # Per-courier summary
        courier_summary = batches.groupby('delivery_user_id').agg(
            total_batches=('batch_id', 'count'),
            dynamic_pickups=('dynamic_pickup_flag', 'sum')
        ).reset_index()
        courier_summary['pct'] = (courier_summary['dynamic_pickups'] / courier_summary['total_batches'] * 100)
        display(courier_summary.sort_values('pct', ascending=False).head(10))


=== Day: 318 ===
Total batches: 496
Dynamic Pickups (Cross-Depot Overlap): 0 (0.00 %)


,delivery_user_id,total_batches,dynamic_pickups,pct
0,00fca617ad52d2deb9650342901a1940,7,0,0.0
1,02aa270595a0d24b1c9d2636fe249ebc,5,0,0.0
2,0624b34f057ea763a1af4721b89d7b7c,4,0,0.0
3,07122388fa861da0c7fc6d07451452e2,1,0,0.0
4,0797e04768928db32df644748c1ce79f,9,0,0.0
5,080c9121cdb81e512181bd93359acdb3,4,0,0.0
6,0b43ad0eea29bfd8e156f60e0f6b066e,6,0,0.0
7,0d4b70c59c4a2840ba9f78d8369a7b49,9,0,0.0
8,0d6bd47d1eea4b6e553ba4fc18b332c7,6,0,0.0
9,0d96b02a9ff59cef57bfad6df87161eb,10,0,0.0



=== Day: 319 ===
Total batches: 491
Dynamic Pickups (Cross-Depot Overlap): 0 (0.00 %)


,delivery_user_id,total_batches,dynamic_pickups,pct
0,00fca617ad52d2deb9650342901a1940,5,0,0.0
1,02aa270595a0d24b1c9d2636fe249ebc,3,0,0.0
2,0624b34f057ea763a1af4721b89d7b7c,4,0,0.0
3,0797e04768928db32df644748c1ce79f,10,0,0.0
4,080c9121cdb81e512181bd93359acdb3,5,0,0.0
5,0d4b70c59c4a2840ba9f78d8369a7b49,8,0,0.0
6,0d6bd47d1eea4b6e553ba4fc18b332c7,6,0,0.0
7,0d96b02a9ff59cef57bfad6df87161eb,8,0,0.0
8,13a2dba21946265cab28f98767d00aa3,4,0,0.0
9,156ba74f4cbb5799f74ba615fa586104,6,0,0.0



=== Day: 320 ===
Total batches: 493
Dynamic Pickups (Cross-Depot Overlap): 0 (0.00 %)


...

,delivery_user_id,total_batches,dynamic_pickups,pct
0,00fca617ad52d2deb9650342901a1940,5,0,0.0
1,02aa270595a0d24b1c9d2636fe249ebc,4,0,0.0
2,02ad3c5104d5605a3808ef675b94b253,5,0,0.0
3,07122388fa861da0c7fc6d07451452e2,1,0,0.0
4,080c9121cdb81e512181bd93359acdb3,4,0,0.0
5,0b43ad0eea29bfd8e156f60e0f6b066e,7,0,0.0
6,0d4b70c59c4a2840ba9f78d8369a7b49,10,0,0.0
7,0d6bd47d1eea4b6e553ba4fc18b332c7,6,0,0.0
8,0d96b02a9ff59cef57bfad6df87161eb,13,0,0.0
9,13a2dba21946265cab28f98767d00aa3,3,0,0.0



=== Day: 331 ===
Total batches: 486
Dynamic Pickups (Cross-Depot Overlap): 0 (0.00 %)


,delivery_user_id,total_batches,dynamic_pickups,pct
0,00fca617ad52d2deb9650342901a1940,11,0,0.0
1,02aa270595a0d24b1c9d2636fe249ebc,3,0,0.0
2,02ad3c5104d5605a3808ef675b94b253,6,0,0.0
3,07122388fa861da0c7fc6d07451452e2,1,0,0.0
4,080c9121cdb81e512181bd93359acdb3,3,0,0.0
5,0b43ad0eea29bfd8e156f60e0f6b066e,10,0,0.0
6,0d4b70c59c4a2840ba9f78d8369a7b49,10,0,0.0
7,0d6bd47d1eea4b6e553ba4fc18b332c7,7,0,0.0
8,0d96b02a9ff59cef57bfad6df87161eb,11,0,0.0
9,13a2dba21946265cab28f98767d00aa3,5,0,0.0



=== Day: 401 ===
Total batches: 518
Dynamic Pickups (Cross-Depot Overlap): 0 (0.00 %)


,delivery_user_id,total_batches,dynamic_pickups,pct
0,00fca617ad52d2deb9650342901a1940,7,0,0.0
1,02aa270595a0d24b1c9d2636fe249ebc,4,0,0.0
2,02ad3c5104d5605a3808ef675b94b253,10,0,0.0
3,07122388fa861da0c7fc6d07451452e2,1,0,0.0
4,080c9121cdb81e512181bd93359acdb3,9,0,0.0
5,0b43ad0eea29bfd8e156f60e0f6b066e,11,0,0.0
6,0d4b70c59c4a2840ba9f78d8369a7b49,10,0,0.0
7,0d96b02a9ff59cef57bfad6df87161eb,11,0,0.0
8,13a2dba21946265cab28f98767d00aa3,5,0,0.0
9,156ba74f4cbb5799f74ba615fa586104,6,0,0.0


## Notes
- This notebook treats orders with identical `receipt_time` (same courier) as part of the same batch, matching the LaDe pipeline's definition.
- Dynamic pickup is detected when a later batch's `receipt_time` occurs while some orders in the earlier batch remain unsigned (`sign_time > next_batch_receipt_time`).
- If `sign_time` is missing or inaccurate, consider using courier GPS traces or delivery status logs to refine the overlap definition.

If you want, I can run this notebook for a specific city file in your environment, or extend it to visualise temporal overlap per courier.